In [1]:
%pip install --upgrade pip
%pip install yfinance torch numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import yfinance as yf
import torch.nn as nn
import numpy as np
import torch 
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timedelta

In [3]:
AAPL = yf.Ticker("AAPL")
AAPL.info

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

In [4]:
AAPL_hist = AAPL.history(period="1mo")
AAPL_hist.info

<bound method DataFrame.info of                                  Open        High         Low       Close  \
Date                                                                        
2026-04-23 00:00:00-04:00  274.796818  275.516157  271.399954  273.178314   
2026-04-24 00:00:00-04:00  272.508933  272.808645  269.401780  270.810486   
2026-04-27 00:00:00-04:00  265.845058  268.112957  264.826008  267.363647   
2026-04-28 00:00:00-04:00  272.089320  272.978515  268.412715  270.460815   
2026-04-29 00:00:00-04:00  267.303712  270.790520  266.794202  269.921326   
2026-04-30 00:00:00-04:00  270.251027  275.745964  267.893213  271.100250   
2026-05-01 00:00:00-04:00  278.603290  286.955610  278.113751  279.882141   
2026-05-04 00:00:00-04:00  279.402577  280.371685  274.606977  276.575165   
2026-05-05 00:00:00-04:00  276.675100  284.308082  276.245503  283.918427   
2026-05-06 00:00:00-04:00  281.660510  287.764872  280.811287  287.245361   
2026-05-07 00:00:00-04:00  289.003717  291.8

In [5]:
ticker = "AAPL"
Seq_len = 60 #number of months for the input window
Hidden_size = 128
num_layers = 2
dropout = .2
batch_size = 32
lr = 1e-3
epochs = 20
Use_log_returns = True
Device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using: {Device}")


using: mps


In [6]:
def fetch_ohlcv(ticker: str, years: int = 5) -> np.array:
    end = datetime.today()
    start = end - timedelta(days = years*365)
    df = yf.download(ticker, start, end, auto_adjust = True, progress = False)
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    return df.values.astype(np.float64), df.index.tolist()
raw, dates = fetch_ohlcv(ticker=ticker)
print(f"Fetched {len(raw)} trading days for {ticker}")

Fetched 1256 trading days for AAPL


In [7]:
def compute_features(raw: np.ndarray) ->np.ndarray:
    if Use_log_returns:
        price = raw[:,:4]
        volume = raw[:,:4:5]

        log_ret = np.log(price[1:] /  price[:-1])
        vol_chg = np.log(volume[1:] /  volume[:-1])

        close_ret = log_ret[:,3]
        rolling_std = np.array([
            close_ret[max(0, i-9): i+1].std()
            for i in range(len(close_ret))
        ]).reshape(-1, 1)

        features = np.hstack([log_ret, vol_chg, rolling_std])
    else:
        scaler = StandardScaler()
        features = scaler.fit.transform(raw)
        features = features[1:]
    return features.astype(np.float32)

features = compute_features(raw)
print(f"feature matrix shape: {features.shape} (timesteps, features)")


feature matrix shape: (1255, 6) (timesteps, features)


In [8]:
class priceSequenceDataset(Dataset):
    def __init__(self, features: np.ndarray, seq_len: int = Seq_len):
        self.features = torch.tensor(features)
        self.seq_len = seq_len
    
    def __len__(self):
        return len(self.features) - self.seq_len
    
    def __getitem__(self,idx):
        x = self.features[idx: idx + self.seq_len]
        y = self.features[idx + self.seq_len, 3]
        return x,y
    
dataset = priceSequenceDataset(features)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last = True)
print(f"Dataset: {len(dataset)} samples | Batches per epoch {len(loader)}")

Dataset: 1195 samples | Batches per epoch 37


In [12]:
input_size = 6 if Use_log_returns else 5
class LSTMPriceEncoder(nn.Module):
    def __init__(self, input_size=input_size, hidden_size=Hidden_size,num_layers=num_layers, dropout=dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout = dropout if num_layers > 1 else 0.0
        )
        self.proj = nn.Linear(hidden_size, hidden_size)
        self.act = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (h_n, _) = self.lstm(x)
        h_last = h_n[-1]
        return self.act(self.proj(h_last))
    
model = LSTMPriceEncoder().to(Device)
print(model)

LSTMPriceEncoder(
  (lstm): LSTM(6, 128, num_layers=2, batch_first=True, dropout=0.2)
  (proj): Linear(in_features=128, out_features=128, bias=True)
  (act): Tanh()
)


In [16]:
head = nn.Linear(Hidden_size, 1).to(Device)
optim = torch.optim.Adam(
    list[any](model.parameters()) + list[any](head.parameters()), lr = lr
)
loss_fn = nn.MSELoss()

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(Device), y.to(Device).unsqueeze(1)
        optim.zero_grad()
        loss = loss_fn(head(model(x)), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optim.step()
        total_loss += loss.item()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{epochs} | MSE loss: {total_loss / len(loader):.6f}")

print("Training complete.")

Epoch   1/20 | MSE loss: 0.000642
Epoch   5/20 | MSE loss: 0.000309
Epoch  10/20 | MSE loss: 0.000318
Epoch  15/20 | MSE loss: 0.000313
Epoch  20/20 | MSE loss: 0.000312
Training complete.


In [20]:
SAVE_PATH = "lstm_encoder_6_features.pt"

torch.save(model.state_dict(), SAVE_PATH)
print(f"Encoder saved → {SAVE_PATH}")

def get_price_feature_vector(model, ticker=ticker, years=1):
    raw, _ = fetch_ohlcv(ticker, years)
    feats  = compute_features(raw)
    if len(feats) < Seq_len:
        raise ValueError(f"Need {Seq_len} days minimum, got {len(feats)}.")
    window = torch.tensor(feats[-Seq_len:]).unsqueeze(0).to(Device)
    model.eval()
    with torch.no_grad():
        return model(window)

price_feature_vector = get_price_feature_vector(model)
print(f"Price feature vector shape: {price_feature_vector.shape}")


Encoder saved → lstm_encoder_6_features.pt
Price feature vector shape: torch.Size([1, 128])
